In [2]:
import pandas as pd

file_path = "Final GCL Rates.xlsx"
raw_df = pd.read_excel(file_path, sheet_name=0, header=None)

print(raw_df.shape)
print(raw_df.head(10))

(43, 73)
                                 0       1        2        3        4   \
0  Unsecured Business Loan Reducing     NaN      NaN      NaN      NaN   
1                         Entry Age   12.00    24.00    36.00    48.00   
2                                18  222.63   400.98   586.71   777.36   
3                                20  232.47   418.20   608.85   801.96   
4                                25  233.70   419.43   610.08   803.19   
5                                30  242.31   440.34   645.75   861.00   
6                                35  287.82   531.36   792.12  1067.64   
7                                40  382.53   722.01  1088.55  1482.15   
8                                45  559.65  1082.40  1654.35  2277.96   
9                                50  926.19  1820.40  2801.94  3872.04   

        5        6        7        8         9   ...       63       64  \
0      NaN      NaN      NaN      NaN       NaN  ...      NaN      NaN   
1       60    72.00    84.00

In [3]:
# Understand the raw grid
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 50)

print(raw_df.iloc[0:15, 0:80])

                                  0        1        2        3         4   \
0   Unsecured Business Loan Reducing      NaN      NaN      NaN       NaN   
1                          Entry Age    12.00    24.00    36.00     48.00   
2                                 18   222.63   400.98   586.71    777.36   
3                                 20   232.47   418.20   608.85    801.96   
4                                 25   233.70   419.43   610.08    803.19   
5                                 30   242.31   440.34   645.75    861.00   
6                                 35   287.82   531.36   792.12   1067.64   
7                                 40   382.53   722.01  1088.55   1482.15   
8                                 45   559.65  1082.40  1654.35   2277.96   
9                                 50   926.19  1820.40  2801.94   3872.04   
10                                55  1531.35  3008.58  4594.05   6280.38   
11                                60  2245.98  4394.79  6664.14   9044.19   

In [4]:
#Clean text
def clean_text(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    return text if text else None

In [5]:
#Split title into product and cover type
def split_product_cover(title):
    title = str(title).strip()

    if "Level Cover" in title:
        product = title.replace("Level Cover", "").strip()
        cover_type = "Level Cover"
    elif "Reducing Cover" in title:
        product = title.replace("Reducing Cover", "").strip()
        cover_type = "Reducing"
    elif "Reducing" in title:
        product = title.replace("Reducing", "").strip()
        cover_type = "Reducing"
    else:
        product = title
        cover_type = None

    return product, cover_type

In [6]:
#Parse age values
def parse_age_band(age_value):
    age_text = str(age_value).strip()

    if "-" in age_text:
        start_age, end_age = age_text.split("-")
        return int(start_age), int(end_age)

    age_num = int(float(age_text))
    return age_num, age_num

In [8]:
#Detect a block start
def is_block_start(df, row, col):
    title = clean_text(df.iloc[row, col])
    next_row_val = clean_text(df.iloc[row + 1, col]) if row + 1 < len(df) else None

    if title and next_row_val == "Entry Age":
        return True
    return False

In [9]:
#Read tenure columns from a block
def extract_tenures(df, header_row, start_col):
    tenures = []
    col = start_col + 1

    while col < df.shape[1]:
        val = df.iloc[header_row, col]

        if pd.isna(val):
            break

        try:
            tenure = int(float(val))
            tenures.append((col, tenure))
            col += 1
        except:
            break

    return tenures

In [10]:
#Read the age rows under a block
def extract_block_rows(df, title_row, start_col):
    product_title = df.iloc[title_row, start_col]
    product, cover_type = split_product_cover(product_title)

    header_row = title_row + 1
    data_start_row = title_row + 2

    tenures = extract_tenures(df, header_row, start_col)
    normalized_rows = []

    row = data_start_row

    while row < df.shape[0]:
        age_cell = df.iloc[row, start_col]

        if pd.isna(age_cell):
            break

        age_text = str(age_cell).strip()

        if age_text == "" or age_text == "Entry Age":
            break

        try:
            age_min, age_max = parse_age_band(age_text)
        except:
            break

        for rate_col, tenure in tenures:
            rate = df.iloc[row, rate_col]

            if pd.notna(rate):
                normalized_rows.append({
                    "product": product,
                    "cover_type": cover_type,
                    "age_min": age_min,
                    "age_max": age_max,
                    "tenure_months": tenure,
                    "rate_per_lakh": float(rate)
                })

        row += 1

    return normalized_rows

In [11]:
#Scan the full workbook
all_rows = []

for col in range(raw_df.shape[1]):
    if is_block_start(raw_df, 0, col):
        block_rows = extract_block_rows(raw_df, 0, col)
        all_rows.extend(block_rows)

In [12]:
all_rows = []

for row in range(raw_df.shape[0] - 1):
    for col in range(raw_df.shape[1]):
        try:
            if is_block_start(raw_df, row, col):
                block_rows = extract_block_rows(raw_df, row, col)
                all_rows.extend(block_rows)
        except:
            pass

In [13]:
#Build the normalized dataframe
normalized_df = pd.DataFrame(all_rows)

print(normalized_df.head(20))
print(normalized_df.shape)

                    product cover_type  age_min  age_max  tenure_months  \
0   Unsecured Business Loan   Reducing       18       18             12   
1   Unsecured Business Loan   Reducing       18       18             24   
2   Unsecured Business Loan   Reducing       18       18             36   
3   Unsecured Business Loan   Reducing       18       18             48   
4   Unsecured Business Loan   Reducing       18       18             60   
5   Unsecured Business Loan   Reducing       18       18             72   
6   Unsecured Business Loan   Reducing       18       18             84   
7   Unsecured Business Loan   Reducing       18       18             96   
8   Unsecured Business Loan   Reducing       18       18            108   
9   Unsecured Business Loan   Reducing       18       18            120   
10  Unsecured Business Loan   Reducing       20       20             12   
11  Unsecured Business Loan   Reducing       20       20             24   
12  Unsecured Business Lo

In [14]:
#Remove duplicates and clean output
normalized_df = normalized_df.drop_duplicates().reset_index(drop=True)

In [16]:
#make sure numeric columns are typed correctly.
normalized_df["age_min"] = normalized_df["age_min"].astype(int)
normalized_df["age_max"] = normalized_df["age_max"].astype(int)
normalized_df["tenure_months"] = normalized_df["tenure_months"].astype(int)
normalized_df["rate_per_lakh"] = normalized_df["rate_per_lakh"].astype(float)

In [22]:
#Save the normalized table
normalized_df.to_csv("normalized_rates.csv", index=False)
normalized_df.to_excel("normalized_rates.xlsx", index=False)

PermissionError: [Errno 13] Permission denied: 'normalized_rates.xlsx'

In [18]:
test = normalized_df[
    (normalized_df["product"] == "Unsecured Business Loan") &
    (normalized_df["cover_type"] == "Reducing") &
    (normalized_df["age_min"] == 18) &
    (normalized_df["age_max"] == 18) &
    (normalized_df["tenure_months"] == 12)
]

print(test)

                   product cover_type  age_min  age_max  tenure_months  \
0  Unsecured Business Loan   Reducing       18       18             12   

   rate_per_lakh  
0         222.63  


In [19]:
test_micro = normalized_df[
    (normalized_df["product"] == "Micro Loan") &
    (normalized_df["cover_type"] == "Reducing") &
    (normalized_df["age_min"] == 18) &
    (normalized_df["age_max"] == 35) &
    (normalized_df["tenure_months"] == 12)
]

print(test_micro)

        product cover_type  age_min  age_max  tenure_months  rate_per_lakh
660  Micro Loan   Reducing       18       35             12         257.07
